# သင်ခန်းစာ ၁၂ - Agent Scratchpad ဖြင့် စကားပြောသမိုင်းလျှော့ချခြင်း

ဤ notebook သည် Microsoft Agent Framework ကို အသုံးပြုပြီး ရှည်လျားသော စကားပြောများတွင် context ကို မည်သို့ စီမံခန့်ခွဲရမည်ကို ပြသသည်။ စကားပြောများ ကြီးထွားလာသည်နှင့်အမျှ token အရေအတွက် မြင့်တက်သွားပြီး ဒီတော့ မော်ဒယ်၏ context ပြတင်းပေါက်ကို ကျော်လွန်သွားနိုင်သည်။ ၎င်းကို ကျွန်ုပ်တို့မှာ **context စုစည်းခြင်း ပုံစံ** နှင့် **agent scratchpad** အား အသုံးပြုပြီး အမြဲတမ်းမှတ်ဉာဏ်ရှိအောင် ဖြေရှင်းထားသည်။

## သင်ယူရမည့်အချက်များ:
1. **Context စီမံခန့်ခွဲမှု အရေးပါမှု**: token ကန့်သတ်ချက်များနှင့် context ပြတင်းပေါက်များကို နားလည်ခြင်း
2. **Context-aware Agents**: ကိုယ်ပိုင် စကားပြော context ကို စီမံခန့်ခွဲသော agents ဖန်တီးခြင်း
3. **Context စုစည်းခြင်း ပုံစံ**: စကားပြော သမိုင်းအတိတ်ကို ပိုမိုတင်းရင်းစေဖို့ ကိရိယာများ အသုံးပြုခြင်း
4. **Agent Scratchpad**: context လျော့ချမှုကို ကျော်လွှားပြီး အမြဲတမ်း အသုံးပြုနိုင်သော မှတ်ဉာဏ်

## မူရင်းလိုအပ်ချက်များ:
- Azure OpenAI ကို ပတ်ဝန်းကျင် အဆင့်များဖြင့် ပြင်ဆင်ထားခြင်း
- ယခင်သင်ခန်းစာများမှ အခြေခံ agent အကြောင်း တတ်မြောက်မှု


## စတင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## အကြောင်းအရာ စီမံခန့်ခွဲမှုသည် အဘယ်ကြောင့် အရေးကြီးသနည်း

LLM တစ်ခုလျှင် ကန့်သတ်ထားသော **context window** တစ်ခုရှိသည် — တစ်ခါတည်းတောင်းဆိုမှုတစ်ခုအတွင်း ပြုလုပ်နိုင်သော token အမြောက်အမြားကန့်သတ်ချက်။ multi-turn စကားပြောဆိုမှု တိုးတက်လာသည့်အချိန်တွင် -

- **Token အရေတွက်သည် အသုံးပြုသူစာတိုက်နှင့် အကူအညီပြန်ကြားချက် တစ်ခုစီနှင့် လိုက်လျောတူညီစွာ တိုးသည်။**
- **Prompt token များသည် ကုန်ကျစရိတ် အကြီးဆုံးဖြစ်သည်** အကြောင်းမှာ တရဲတိုက် အတိတ်ဒေတာအားလုံးအား ဆက်တိုက်ပို့ပေးရသောကြောင့်ဖြစ်သည်။
- နောက်ဆုံးတွင် စကားပြောဆိုမှုသည် **context window ကိုကျော်လွန်၍** မော်ဒယ်သည် အတိုကောက်မှတ်တမ်းပြုလုပ်ရန် သို့မဟုတ် အမှားဖြစ်နိုင်သည်။

### အကြောင်းအရာ စီမံခန့်ခွဲမှုအတွက် နည်းလမ်းများ

| နည်းလမ်း | မည်သို့ အလုပ်လုပ်သည် | ရရှိနိုင်သည့်သက်ရောက်မှု |
|---|---|---|
| **အတိုချုပ်ခြင်း** | အဟောင်းဆုံးစာတိုက်များကို ဖယ်ရှားသည် | အစပိုင်းအကြောင်းအရာ ပျောက်ဆုံးသွားသည် |
| **စာချုပ်ရေးခြင်း** | အဟောင်းစာတိုက်များကို တိုတောင်း စာချုပ်တစ်ခုအဖြစ် ဖော်ပြသည် | အချို့အသေးစိတ်ပျောက်ဆုံးသော်လည်း အချက်အလက် အဓိကကို ထိန်းသိမ်းပေးသည် |
| **Scratchpad / အပြင်မှ မှတ်ဉာဏ်** | အဓိက အချက်အလက်များကို စကားပြောဆိုမှုအပြင် သိမ်းဆည်းထားသည် | ကိရိယာခေါ်ယူမှု လိုအပ်သော်လည်း လျော့နည်းမှုများကို ရှောင်ရှားနိုင်သည် |

ဤ notebook တွင် ကျွန်ုပ်တို့သည် **စာချုပ်ရေးခြင်း** နှင့် **scratchpad ကိရိယာ** ကို ပေါင်းစပ်အသုံးပြု၍ စကားပြောဆိုမှုသမိုင်းကို တိုချုပ်သော်လည်း အဆက်မပြတ်စဉ်ဆက်မယ့် သက်တမ်းကို ထိန်းသိမ်းနိုင်စေရန်လုပ်ဆောင်ထားသည်။


## အေၾကာင္းအရာသိရွိၿပီး အေးဂျင့် တည်ဆောက်ခြင်း


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## ကြာရှည်သော စကားပြောချိန်ကို စမ်းသပ်ကြည့်ခြင်း

အကြောင်းအရာ သိမြင်မှု ဘယ်လို စုပေါင်းတိုးတက်သွားမလဲ စမ်းသပ်ဖို့ မျိုးစုံပြောဆိုမှုတစ်ခုကို လိုက်လျောညီထွေ ကြည့်ကြမယ်။ အသံခံ ကိုယ်စားလှယ်က အဓိကအသေးစိတ်များ (အကြိုက်များ၊ ဘတ်ဂျက်၊ ခရီးသွားရက်များ) ကို အကြောင်းအရာများပြန်လည်ထိန်းသိမ်းပြီး ဆက်လက်ဆောင်ရွက်မှု ပြသနိုင်ဖို့ ဖြစ်ရမယ်။


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

အေးဂျင့်က ကြိုတင်ပြောခဲ့သော အကြောင်းအရာများကို မေ့မလျော့ထားပုံကို သတိပြုပါ — ဂျပန်, ဆူရှီ, ဘုရားကျောင်းများ, ဓာတ်ပုံကူးခြင်း, ၃,၀၀၀ ဒေါ်လာ ဘတ်ဂျက်, တစ်ဦးချင်း ခရီးသွားခြင်းနဲ့ ဧပြီလကာလအကြောင်းကို သိရှိထားပါတယ်။ စကားပြောဆိုမှုတိုတောင်းတဲ့အခါ ဒီနည်းလမ်းက အဆင်ပြေပါပေမယ့် စကားပြောဆိုမှု ပိုများလာတာနဲ့အမျှ အစီအစဉ်အပြည့်အစုံကို ထပ်မံပို့ရဖို့ အကုန်ကျစရိတ် မြင့်လာပါတယ်။

အခု စကားပြောဆိုမှုကို အပို Turn များဖြင့် ဆက်လက်လုပ်ဆောင်ကြည့်ပြီး ဆက်စပ်မှုစုဆောင်းမှုကို ကြည့်ရအောင်။


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

စကားပြောဆိုမှုတိုးလာသည်နှင့်အမျှ၊ စုစည်းထားသော စာသားအကြောင်းအရာကို တောင်တိမ်းသော ပုံစံသို့ စုစည်းရန် **အနှစ်ချုပ်မှုကိရိယာ** ကို အသုံးပြုနိုင်ပါသည်။ အဲ့ဒီအေးဂျင့်သည် အဆိုပါကိရိယာကို ခေါ်ယူ၍ အရေးကြီးသောနှစ်သက်ချက်များကို မှတ်တမ်းတင်ပေးသည့်အတွက် အဟောင်းစာသားများ မပါရှိတော့လျှင်တောင် အရေးကြီးသောအချက်အလက်များကို ထိန်းသိမ်းထားနိုင်ပါသည်။

ဤနမူနာသည် ပိုမိုတီထွင်ပြီးသော သမိုင်းနောက်ခံလျော့ချမှုအတွက် အခြေခံတစ်ခုဖြစ်သည်။
1. အေးဂျင့်သည် စကားပြောဆိုမှုမှ အချက်အလက်အဓိကများကို ခွဲခြားသတ်မှတ်သည်
2. အနှစ်ချုပ်မှုကိရိယာကို ခေါ်ယူ၍ အချက်အလက်များကို သိမ်းဆည်းထားသည်
3. အဟောင်းစာသားများကို လုံခြုံစွာ ဖယ်ရှားနိုင်သည်၊ အကြောင်းမှာ အနှစ်ချုပ်သည် အရေးကြီးသော အချက်များကို ဖမ်းဆီးထားသည်။

အောက်တွင် အေးဂျင့်သည် ယူဆထားသည့် အချက်အလက်များကို အနှစ်ချုပ်အတိုအထွာ သတ်မှတ်ရန် ခေါ်ယူနိုင်သည့် `summarize_preferences` ကိရိယာကို သတ်မှတ်ထားပါသည်။


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework ကို အသုံးပြုပြီး ရေရှည် အေးဂျင့် အပြန်အလှန် ပြောဆိုမှုတွေမှာ context ကို မန်နေဂျ් လုပ်နည်းကို သင်ယူခဲ့ပါတယ်။

### အဓိက အတွေးအခေါ်များ
- **Context window စာတန်းကန် အတိုင်းအတာ ရှိသည်** — ပြောဆိုမှု မှတ်တမ်းထဲရှိ token တစ်ခုချင်းစီမှာ ကုန်ကျစရိတ်ရှိပြီး အကန့်အသတ်ကို ထိခိုက်စေသည်။
- **ကိုယ်စုစုပေါင်းအနှစ်ချုပ် ကိရိယာများ** သည် agent ကို စုဆောင်းထားသော context ကို သေးငယ်စေ့စပ်သော အနှစ်ချုပ်အဖြစ် ပေါင်းစည်းနိုင်စေပြီး token အသုံးပြုမှုကို လျှော့နည်းစေသည့်အပြင် အရေးပါသော သတင်းအချက်အလက်များကို ထိန်းသိမ်းထားပေးသည်။
- **Agent scratchpads** သည် မည်သည့်ပြောဆိုမှု လျော့နည်းမှု ဖြစ်လာပေမယ့် အသီးသီး ယနေ့၌ မျှတသော ပြင်ပမှတ်ဉာဏ် ဖြစ်စေသည်။

### သင် လုပ်ဆောင်ခဲ့သည်များ
- များစွာသော ပြောဆိုမှုများတွင် ဆက်လက်မှု ယုံကြည်စိတ်ချမှု ရှိသော **context-aware agent** တစ်ခု
- အဓိက သုံးစွဲသူ အချက်အလက်များကို အနှစ်ချုပ် သေချာဖော်ပြရှိသည့် **အနှစ်ချုပ်ကိရိယာ** (`summarize_preferences`)
- context ထိန်းသိမ်းမှုနှင့် ပြောင်းလဲမှုကို ပြသသည့် **ပြောဆိုမှု များစွာ** တစ်ခု

### အမှန်တကယ် အသုံးချနိုင်မှုများ
- **ဖောက်သည်ဝန်ဆောင်မှု Bot များ**: ရေရှည်အထောက်အပံ့အစည်းအဝေးများအတွင်း မြင်သာမှုများကို မှတ်မိထားနိုင်ခြင်း
- **ပုဂ္ဂိုလ်ရေး အကူအညီပေးများ**: context ကို ထပ်မဖော်ပြဖို့ ပရောဂျက်များကို ဆက်လက်သိရှိနေစေခြင်း
- **ပညာရေး ဆရာများ**: ကျောင်းသား တိုးတက်မှုများကို အချိန်ကြာမြင့်စွာ ဆက်လက်ထိန်းသိမ်းခြင်း

### နောက်တစ်ဆင့်များ
- ဖိုင် အခြေပြု ထိန်းသိမ်းမှုရှိသော အပြည့်အစုံ Scratchpad ကိရိယာ တည်ဆောက်ခြင်း
- အနှစ်ချုပ် ပြုလုပ်ပြီးနောက် အလိုအလျောက် ရှေးလွှာ ရှင်းလင်းခြင်း ထည့်သွင်းခြင်း
- စာရင်းနှင့် ဗက်တာ ဒေတာဘေ့စ်များ ဖြင့် Semantic မှတ်ဉာဏ် ရှာဖွေမှု ပေါင်းစည်းခြင်း
- ရက်ပေါင်းများစွာ အလိုတော်မက် စကားဝိုင်းများကို context အပြည့်အစုံဖြင့် ပြန်လည်ဆက်လက် ပြုလုပ်နိုင်သော agent များ တည်ဆောက်ခြင်း


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပယ်ချခြင်း**:  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြုကာ ဘာသာပြန်ထားပါသည်။ ကျနော်တို့သည် တိကျမှန်ကန်မှုအတွက် စိုးရိမ်ကြိုးပမ်းသော်လည်း၊ အလိုအလျောက် ဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှန်ကန်မှုမရှိမှုများ ပါဝင်နိုင်ကြောင်း သိရှိထားရန် တောင်းဆိုပါသည်။ မူရင်းစာတမ်း၏ မိဘစကားဖြစ်သည့်ဘာသာစကားသည် အာဏာပိုင်ရင်းမြစ်အဖြစ် ယူဆရပါမည်။ အရေးကြီးသောအချက်အလက်များအတွက် ပညာရှင်လူ့ဘာသာပြန်ခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်မှု အသုံးပြုမှုမှ ဖြစ်ပေါ်လာနိုင်သည့် ရိုက်ထွင်းမှုများ သို့မဟုတ် မှားအပ်မှုများအတွက် ကျနော်တို့၏ တာဝန်မရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
